<div style="text-align: justify; max-width: 90ch">

# Lecture 02d: Chroma, retrievers and the LangChain RAG chain

**Status: Mandatory reading.**

`lec_02b` searched a NumPy matrix of four lectures, rebuilt from a cache file. A real system keeps its vectors in a **vector store**,
which saves them with their text and metadata, and searches and filters them fast. [Chroma](https://docs.trychroma.com/) is an open-source vector store that runs inside Python and keeps its data in a folder.
On top of it we build the LangChain RAG chain, then ask what every RAG system must answer: does retrieval find the right notes?

**What this notebook covers**

- A persistent Chroma index of all thirteen lectures.
- Ingesting once: why re-running a notebook must not double the index.
- Search with scores, a metadata filter and a content filter.
- The LangChain RAG chain, and what MLflow autolog records.
- Measuring retrieval with **hit rate** and **MRR** on an evaluation set.

</div>

In [1]:
import json
import os
import time
from pathlib import Path

import mlflow
import pandas as pd
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_ollama import ChatOllama, OllamaEmbeddings

from llm_course.checks import check_mlflow
from llm_course.corpus import load_corpus
from llm_course.rag import split_documents

In [2]:
# --- Course configuration: edit TIER only; settings in .env (Read: 01d_env_hugging_face) ---
TIER = "cpu"  # 8-16 GB RAM, no GPU (default)
# TIER = "gpu"  # >= 8 GB VRAM: bigger, better answers (pull qwen3:8b first)

REPO = Path.cwd().resolve().parents[1]  # runs from a lecture subfolder
load_dotenv(REPO / ".env")  # load your .env settings: OLLAMA_HOST, HF_TOKEN, ...
TIER = os.environ.get("COURSE_TIER", TIER)  # tools/run_all.py can override
from huggingface_hub.utils import disable_progress_bars

disable_progress_bars()  # no download bars in saved outputs (works after the imports)
CHAT_MODEL = {"cpu": "qwen3:1.7b", "gpu": "qwen3:8b"}[TIER]
EMBED_MODEL = "nomic-embed-text"
OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
MLFLOW_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://127.0.0.1:5010")
# To override .env in this notebook only, uncomment and edit (never a token):
# OLLAMA_HOST = "http://192.168.1.20:11434"
# MLFLOW_URI = "http://127.0.0.1:5011"
EXPERIMENT = "llm-course-02-rag"
DATA_DIR = REPO / "data"  # gitignored runtime state (indexes, caches)

from llm_course.checks import check_ollama

check_ollama(OLLAMA_HOST, [CHAT_MODEL])

Ollama OK at http://localhost:11434; models ready: qwen3:1.7b


<div style="text-align: justify; max-width: 90ch">

**Rebuild inputs.** All thirteen lectures, cut into chunks of 800 characters with an overlap of 100 (`lec_02c`), in two lines.

</div>

In [3]:
documents = load_corpus(REPO / "corpus" / "uoa_py_course")

chunks = split_documents(documents, chunk_size=800, chunk_overlap=100)

print(len(documents), "files ->", len(chunks), "chunks")

125 files -> 2452 chunks


<div style="text-align: justify; max-width: 90ch">

## 1. A vector store

| Term | Meaning |
| --- | --- |
| **vector store** | a database for vectors: stores each one with its text and metadata, and returns the nearest to a query |
| **collection** | one named set of vectors inside the store, like a table |
| **id** | the unique name of one stored chunk |
| **persist directory** | the folder where Chroma keeps a collection between sessions, here `data/chroma/` |
| **index** | a structure that finds near neighbours without comparing the query with every vector; Chroma's is [HNSW](https://en.wikipedia.org/wiki/Hierarchical_navigable_small_world) |
| **retriever** | LangChain's name for anything that returns the documents for a question |

Chroma computes embeddings through an **embedding function**: `OllamaEmbeddings`, LangChain's wrapper around the same `nomic-embed-text`.
`"hnsw:space": "cosine"` makes the index compare vectors by cosine, as in `lec_02a`.

</div>

In [4]:
check_ollama(OLLAMA_HOST, [EMBED_MODEL])

embeddings = OllamaEmbeddings(model=EMBED_MODEL, base_url=OLLAMA_HOST)

store = Chroma(
    collection_name="course_notes",
    embedding_function=embeddings,
    persist_directory=str(DATA_DIR / "chroma"),
    collection_metadata={"hnsw:space": "cosine"},
)

print(len(store.get(include=[])["ids"]), "chunks stored so far")

Ollama OK at http://localhost:11434; models ready: nomic-embed-text
2452 chunks stored so far


<div style="text-align: justify; max-width: 90ch">

## 2. Ingest once

Adding documents embeds them and writes them to disk. But a notebook runs many times. The mistake, on a throwaway collection kept in memory:

</div>

In [5]:
# WRONG: add the documents every time the cell runs
scratch = Chroma(
    collection_name="scratch", embedding_function=embeddings
)  # in memory only

for run in [1, 2]:
    scratch.add_documents(chunks[:5])
    print(f"after run {run}: {len(scratch.get(include=[])['ids'])} chunks")

scratch.delete_collection()  # clean up the demonstration

after run 1: 5 chunks


after run 2: 10 chunks


<div style="text-align: justify; max-width: 90ch">

Each run stores the five chunks again under new random ids. After ten runs of the notebook the index would hold every chunk ten times,
and a search would return four copies of the same text.

</div>

In [6]:
# RIGHT: embed only when the collection does not hold exactly these chunks, under stable ids
if len(store.get(include=[])["ids"]) != len(chunks):
    store.reset_collection()  # empty, half-built or out of date: start again
    started = time.time()

    for start in range(0, len(chunks), 256):
        batch = chunks[start : start + 256]
        ids = [f"{c.metadata['source']}#{c.metadata['start_index']}" for c in batch]
        store.add_documents(batch, ids=ids)
        print(f"{start + len(batch)} / {len(chunks)}")

    print(f"embedded in {time.time() - started:.0f} s")

print(len(store.get(include=[])["ids"]), "chunks in data/chroma")

2452 chunks in data/chroma


<div style="text-align: justify; max-width: 90ch">

The count check compares with the chunks we expect: an empty, half-built or out-of-date collection is rebuilt, a complete one is reused, so every later run is instant.
The ids (file and start position) stay the same while the file does, so adding a chunk again replaces it instead of duplicating it.
One change the count cannot see is a new embedding model: after changing `EMBED_MODEL`, delete `data/chroma/`. The same code is promoted as `llm_course.rag.build_chroma_index`.

</div>

<div style="text-align: justify; max-width: 90ch">

## 3. Search with scores

`similarity_search_with_score` returns the nearest chunks with their **distance**: in this collection, 1 − cosine similarity, so **lower is closer**.

</div>

In [7]:
def show(results):
    """One row per (document, distance) pair returned by the store."""
    return pd.DataFrame(
        [
            {
                "distance": round(distance, 3),
                "source": doc.metadata["source"],
                "kind": doc.metadata["kind"],
                "heading": doc.metadata["heading"][:45],
            }
            for doc, distance in results
        ]
    )


show(store.similarity_search_with_score("Which lecture covers data leakage?", k=4))

,distance,source,kind,heading
0,0.300,lecture_12/lec_12c_logistic_regression_assumpt...,lecture,Lecture map
1,0.326,lecture_08/lec_08e_example_EDA_heart_disease.md,lecture,Lecture 08e: Exploratory Data Analysis — Hear
2,0.332,lecture_12/lec_12d_logistic_regression_hyperpa...,lecture,Lecture map
3,0.341,lecture_11/lec_11f_regression_advanced_topics.md,lecture,Lecture 11f — Advanced regression topics


<div style="text-align: justify; max-width: 90ch">

Check each row: does the chunk explain leakage, or does it only share words such as "lecture" and "data" with the question?
Similarity measures closeness to the question's words and topic, not whether a chunk answers it. Now `lec_02b`'s two questions, on chunks.
First the one whose answer the 4,000-character cut lost:

</div>

In [8]:
HEART_QUESTION = "In the heart-disease screening example, what recall does the model reach at threshold 0.3?"

show(store.similarity_search_with_score(HEART_QUESTION, k=4))

,distance,source,kind,heading
0,0.242,lecture_12/lec_12a_logistic_regression_intuiti...,lecture,Lower the decision threshold — different conf
1,0.254,lecture_12/lec_12a_logistic_regression_intuiti...,lecture,Lower the decision threshold — different conf
2,0.262,lecture_12/lec_12a_logistic_regression_intuiti...,lecture,§E. A small real binary example — heart-disea
3,0.270,lecture_13/lec_13a_gridsearchcv_hyperparameter...,lecture,Choosing the scoring metric


<div style="text-align: justify; max-width: 90ch">

Look for `lec_12a` among the sources: its threshold table now sits in a chunk of its own, no longer cut off, so when retrieval finds it,
the answer can reach the prompt. Then the class name:

</div>

In [9]:
show(store.similarity_search_with_score("What does GridSearchCV do?", k=4))

,distance,source,kind,heading
0,0.222,lecture_13/lec_13a_gridsearchcv_hyperparameter...,lecture,`GridSearchCV`: search a grid automatically
1,0.269,lecture_12/lec_12d_logistic_regression_hyperpa...,lecture,Lecture 12d — Logistic Regression: Hyperparam
2,0.281,lecture_13/lec_13_exercises_solutions.md,solutions,Section 1 - Hyperparameter search
3,0.292,lecture_13/goals_13.md,goals,Required


<div style="text-align: justify; max-width: 90ch">

No empty headings any more: `lec_02c` dropped the pieces under 80 characters. Whether `lec_13a`, the notebook on `GridSearchCV`, ranks first depends on the embedding model and its version:
to the model the class name is still a few meaningless fragments (`lec_02c`). Section 4 makes that search reliable.

</div>

<div style="text-align: justify; max-width: 90ch">

## 4. Filters

Every chunk carries its file's `kind` (`lec_02c`):

- lecture notes
- setup guides
- exercises
- solutions
- goals
- "read_agents" readings

The last four restate the lectures in fewer words and may crowd the results; section 6 measures whether they do. A **metadata filter** keeps only the lecture notes and guides;
`{"kind": {"$in": [...]}}` is Chroma's [filter language](https://docs.trychroma.com/docs/querying-collections/metadata-filtering) for "kind is one of these values":

</div>

In [10]:
LECTURE_NOTES = {"kind": {"$in": ["lecture", "guide"]}}

show(
    store.similarity_search_with_score(
        "What does GridSearchCV do?", k=4, filter=LECTURE_NOTES
    )
)

,distance,source,kind,heading
0,0.222,lecture_13/lec_13a_gridsearchcv_hyperparameter...,lecture,`GridSearchCV`: search a grid automatically
1,0.269,lecture_12/lec_12d_logistic_regression_hyperpa...,lecture,Lecture 12d — Logistic Regression: Hyperparam
2,0.319,lecture_13/lec_13b_pipelines_preprocessing_mod...,lecture,6. `GridSearchCV` over a pipeline: the `step_
3,0.341,lecture_13/lec_13a_gridsearchcv_hyperparameter...,lecture,Recap and what's next


<div style="text-align: justify; max-width: 90ch">

The filter removes the readings and exercises, but it cannot make the embedding understand a class name: which chunks rank first still depends on the run.
When a question names an exact term, add a **content filter**: `where_document={"$contains": ...}` keeps only chunks whose text contains the term, and the embedding ranks those.

</div>

In [11]:
show(
    store.similarity_search_with_score(
        "What does GridSearchCV do?",
        k=4,
        filter=LECTURE_NOTES,
        where_document={"$contains": "GridSearchCV"},
    )
)

,distance,source,kind,heading
0,0.222,lecture_13/lec_13a_gridsearchcv_hyperparameter...,lecture,`GridSearchCV`: search a grid automatically
1,0.269,lecture_12/lec_12d_logistic_regression_hyperpa...,lecture,Lecture 12d — Logistic Regression: Hyperparam
2,0.319,lecture_13/lec_13b_pipelines_preprocessing_mod...,lecture,6. `GridSearchCV` over a pipeline: the `step_
3,0.341,lecture_13/lec_13a_gridsearchcv_hyperparameter...,lecture,Recap and what's next


<div style="text-align: justify; max-width: 90ch">

Every result now contains the exact word `GridSearchCV`, and the embedding ranks only those chunks. If the table matches the one above, the embedding had already ranked such chunks first on this run:
the filter guarantees the term, not a different ranking. Exact matching and meaning complement each other; production systems combine them automatically, which is called **hybrid search**.

### ⏸ Pause point

When would a content filter hurt? Think of a misspelled class name, or a question that describes the idea without ever naming it.

</div>

<div style="text-align: justify; max-width: 90ch">

## 5. The LangChain RAG chain

`lec_01d` chained prompt, model and parser. A RAG chain adds a first step:

- The **retriever** fetches four chunks.
- `format_context` turns them into text labelled with their sources.
- `RunnablePassthrough` forwards the question unchanged.

The dict at the start runs both branches on the same input. A plain function such as `format_context` becomes a chain step as soon as you pipe into it:
LangChain wraps it for you.

</div>

In [12]:
retriever = store.as_retriever(search_kwargs={"k": 4, "filter": LECTURE_NOTES})


def format_context(docs):
    """Turn retrieved Documents into the text block the prompt receives."""
    return "\n\n".join(
        f"[source: {d.metadata['source']}]\n{d.page_content}" for d in docs
    )

In [13]:
prompt = ChatPromptTemplate.from_template(
    "You answer questions about a Python course using only the notes below. "
    "If the notes do not contain the answer, say so. Mention the source file you used.\n\n"
    "Notes:\n{context}\n\nQuestion: {question}"
)

llm = ChatOllama(
    model=CHAT_MODEL,
    base_url=OLLAMA_HOST,
    reasoning=False,
    temperature=0,
    num_predict=200,
    num_ctx=8192,
)

rag_chain = (
    {"context": retriever | format_context, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

<div style="text-align: justify; max-width: 90ch">

`lec_02b` built its spans by hand with `@mlflow.trace`. LangChain reports its own steps to MLflow: one line, [`mlflow.langchain.autolog()`](https://mlflow.org/docs/latest/genai/tracing/integrations/listing/langchain/),
traces every chain from here on. The tracking server must be running (guide `02a_mlflow_server`).

</div>

In [14]:
check_mlflow(MLFLOW_URI)
mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT)
mlflow.tracing.disable_notebook_display()  # keep the trace viewer out of the saved notebook
mlflow.langchain.autolog()

print(rag_chain.invoke("Which lecture covers pandas?"))

MLflow OK at http://127.0.0.1:5010


The lecture that covers pandas is **lecture_07a_pandas_df_index_slice.md**. 

This lecture discusses:
- Opening a CSV file and reading data as a pandas DataFrame.
- Basic DataFrame methods, attributes, and descriptive statistics.
- Columns and index structure.
- Selecting data using `.loc` and `.iloc`.
- Transposing a DataFrame.

The question "Which lecture covers pandas?" is answered by the source file **lecture_07a_pandas_df_index_slice.md**.


In [15]:
trace = mlflow.get_trace(mlflow.get_last_active_trace_id(), flush=True)

print(pd.DataFrame([{"span": s.name, "type": s.span_type} for s in trace.data.spans]))

                                 span        type
0                    RunnableSequence       CHAIN
1  RunnableParallel<context,question>       CHAIN
2                    RunnableSequence       CHAIN
3                VectorStoreRetriever   RETRIEVER
4                 RunnablePassthrough       CHAIN
5                      format_context       CHAIN
6                  ChatPromptTemplate       CHAIN
7                          ChatOllama  CHAT_MODEL
8                     StrOutputParser       CHAIN


In [16]:
print("tokens:", trace.info.token_usage)

tokens: {'input_tokens': 730, 'output_tokens': 102, 'total_tokens': 832}


<div style="text-align: justify; max-width: 90ch">

Read the answer critically: is the file it cites among the four retrieved chunks, spelled as in their metadata, and does that chunk say what the answer claims?
A model can cite a file it never received, or a file named inside a chunk; only the trace shows what retrieval actually handed over.

### ⏸ Pause point

Open the newest trace in the MLflow UI. The `VectorStoreRetriever` span lists the four chunks, the `ChatOllama` span shows the prompt that carried them.
Which step would you fix? Everything you built by hand in `lec_02b` is recorded without a single decorator, plus the **token usage**:
the number a hosted provider would bill.

</div>

<div style="text-align: justify; max-width: 90ch">

## 6. Measuring retrieval

One good answer proves nothing. We need many questions with known answers: `corpus/eval/qa_eval_set.jsonl` holds 19 questions about the notes,
one JSON object per line, each with the file that answers it (`expected_source`) and the facts a good answer states (lecture 3 uses those).

</div>

In [17]:
eval_path = REPO / "corpus" / "eval" / "qa_eval_set.jsonl"

eval_set = [
    json.loads(line) for line in eval_path.read_text(encoding="utf-8").splitlines()
]

pd.DataFrame(eval_set)[["id", "question", "expected_source"]].head()

,id,question,expected_source
0,q01,Which lecture explains data leakage in preproc...,lecture_13/lec_13b_pipelines_preprocessing_mod...
1,q02,What does GridSearchCV do?,lecture_13/lec_13a_gridsearchcv_hyperparameter...
2,q03,When should I use RandomizedSearchCV instead o...,lecture_13/lec_13a_gridsearchcv_hyperparameter...
3,q04,How do I log a metric in an MLflow run?,lecture_13/lec_13c_mlflow_tracking_basics.md
4,q05,How can MLflow record a whole hyperparameter s...,lecture_13/lec_13e_mlflow_logging_sweeps.md


<div style="text-align: justify; max-width: 90ch">

Two numbers summarise retrieval:

| Metric | The question it answers |
| --- | --- |
| **hit rate@k** | for what share of the questions is a chunk of the expected file among the top `k`? |
| **MRR**, mean reciprocal rank | on average, how high is the first right chunk? 1 if always first, 0.5 if always second, 0 if not in the top 8 |

</div>

In [18]:
def source_rank(docs, expected_source):
    """Position (1 = first) of the first chunk from the expected file, or None."""
    for position, doc in enumerate(docs, start=1):
        if doc.metadata["source"] == expected_source:
            return position
    return None


rows, ranks_by_search = [], {}

for label, search_filter in [
    ("all chunks", None),
    ("lecture notes only", LECTURE_NOTES),
]:
    ranks = [
        source_rank(
            store.similarity_search(q["question"], k=8, filter=search_filter),
            q["expected_source"],
        )
        for q in eval_set
    ]
    row = {"search": label}
    for k in [1, 2, 4, 8]:
        row[f"hit@{k}"] = sum(r is not None and r <= k for r in ranks) / len(ranks)
    row["MRR"] = sum(1 / r for r in ranks if r) / len(ranks)
    rows.append(row)
    ranks_by_search[label] = ranks

pd.DataFrame(rows).round(2)

,search,hit@1,hit@2,hit@4,hit@8,MRR
0,all chunks,0.84,0.95,1.0,1.0,0.91
1,lecture notes only,0.95,0.95,1.0,1.0,0.96


<div style="text-align: justify; max-width: 90ch">

Compare the two rows: does the filter lift the top ranks (hit@1, MRR)? Does a larger `k` beyond 4 still add much? Every question whose file is not in the top four is listed below;
the list changes with the model version, and may be empty:

</div>

In [19]:
for q, rank in zip(eval_set, ranks_by_search["lecture notes only"]):
    if rank is None or rank > 4:
        print(q["id"], rank, q["question"])

<div style="text-align: justify; max-width: 90ch">

In example runs the misses, when there were any, were the questions that name a class: `GridSearchCV` (q02) and `RandomizedSearchCV` (q03).
With the content filter of section 4, each on its own class name:

</div>

In [20]:
EXACT_TERMS = {"q02": "GridSearchCV", "q03": "RandomizedSearchCV"}

for q in eval_set:
    if q["id"] in EXACT_TERMS:
        docs = store.similarity_search(
            q["question"],
            k=8,
            filter=LECTURE_NOTES,
            where_document={"$contains": EXACT_TERMS[q["id"]]},
        )
        print(
            q["id"],
            "rank with the content filter:",
            source_rank(docs, q["expected_source"]),
        )

q02 rank with the content filter: 1
q03 rank with the content filter: 1


<div style="text-align: justify; max-width: 90ch">

In example runs both ranked in the top four with the content filter. The course uses **`k=4` with the lecture filter**: in example runs, the right file for 17 to 19 of the 19 questions, at under 1,000 tokens of notes per prompt.
Three limits of these numbers:

- **Few questions**: one question moves the hit rate by 5 points.
- **One expected file per question**: a search that finds a goals or exercise file that also answers is scored as a miss,
  part of why "all chunks" can score lower.
- **Choices tuned on the test**: the same 19 questions chose `k` and the filter, and will grade the system in lecture 3.
  That is the leakage of `lec_13b` (question q01!) in a new form: a larger project keeps separate questions for choosing and for grading.

They also measure retrieval only; whether the *answer* is right is lecture 3.

</div>

<div style="text-align: justify; max-width: 90ch">

`source_rank`, `format_context` and the ingestion guard are promoted to `llm_course.rag` (with `hit_rate_and_mrr` and `make_rag_chain`),
ready for lecture 3.

</div>

<div style="text-align: justify; max-width: 90ch">

## Recap

- A **vector store** keeps vectors with their text and metadata in a named **collection**, and returns the nearest ones to a query.
- Ingest **idempotently**: check the count before embedding, and give chunks stable ids.
- A **metadata filter** removes whole kinds of documents; a **content filter** adds an exact-word condition the embedding cannot provide.
- The RAG chain is `{"context": retriever | format_context, "question": RunnablePassthrough()} | prompt | llm | parser`,
  and autolog traces it with token usage.
- **Hit rate@k** and **MRR** on an evaluation set with expected sources turn "it seems to work" into numbers you can compare.

**Next: lecture 3**, which judges the answers themselves, not only the retrieval behind them.

</div>